# FastSpeech2 Telugu TTS Pipeline (IndicTTS Telugu dataset)



**Steps in this notebook**
1. Set up Colab (GPU, Google Drive, code, libraries)
2. Load the **IndicTTS Telugu** dataset (IIT Madras recordings, one female and one male speaker)
3. Clean the Telugu text and pick ~5 hours for training + ~1 hour for validation (no overlap)
4. Find *when* each sound is spoken (forced alignment with MFA)
5. Look at the repo's `Telugu` folder (needed before preprocessing and training)



## 1. Check GPU



In [ ]:
!nvidia-smi

Fri Nov  6 08:52:39 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   52C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Mount Google Drive and create folders



In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
for p in ["raw_data", "preprocessed_data", "output/Telugu", "backup"]:
    os.makedirs(f"/content/drive/MyDrive/fs2/{p}", exist_ok=True)
print("Folders ready")

Mounted at /content/drive
Folders ready


## 3. Clone the project and look at the Telugu folder



In [ ]:
!git clone https://github.com/Narmadas-31/fastspeech2-english-telugu.git /content/project
%cd /content/project
!ls
print("---- Telugu folder ----")
!ls /content/project/Telugu

Cloning into '/content/project'...
remote: Enumerating objects: 104, done.
remote: Counting objects: 100% (104/104), done.
remote: Compressing objects: 100% (78/78), done.
remote: Total 104 (delta 25), reused 99 (delta 22), pack-reused 0 (from 0)
Receiving objects: 100% (104/104), 1.39 MiB | 11.61 MiB/s, done.
Resolving deltas: 100% (25/25), done.
/content/project
English  LICENSE  README.md  Report  requirements.txt  Results	Telugu
---- Telugu folder ----
audio	     dataset.py    lexicon	  requirements_ref.txt	transformer
checkpoints  evaluate.py   model	  synthesize.py		utils
config_ref   hifigan	   preprocessor   text
config.yaml  inference.py  preprocess.py  train.py


## 4. Install libraries

A *library* is ready-made code that someone else wrote (for audio, numbers, etc.). `pip install` downloads them.
`datasets<4` is the same version we used for English; it works smoothly with audio data.

In [ ]:
!pip install -q "datasets<4" huggingface_hub soundfile librosa pyyaml tgt inflect pyworld tensorboard matplotlib pandas scipy tqdm

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.12.0 requires fsspec==2025.12.0, but you have fsspec 2025.3.0 which is incompatible.


## 5. Load the IndicTTS Telugu dataset

**What is it?** IndicTTS is a studio-quality speech database made by IIT Madras. We use the Hugging Face copy `SPRINGLab/IndicTTS_Telugu`: Telugu sentences read by **1 female and 1 male** speaker, about 8.7 hours in total, recorded at 48 kHz.

**License:** the data follows the *License For Use of Indic TTS* (IIT Madras). Read it once, and mention IndicTTS / IIT Madras as the source in your README and report.

The first run downloads about 6.4 GB to Colab's disk. This may take several minutes, which is normal.


In [ ]:
import io, os, re, json, random, collections, unicodedata
import numpy as np, soundfile as sf, librosa
from datasets import load_dataset, Audio

ds = load_dataset("SPRINGLab/IndicTTS_Telugu", split="train")
print(ds)
print(ds.features)

README.md:   0%|          | 0.00/3.25k [00:00<?, ?B/s]

data/train-00000-of-00004.parquet: reconstructing file:   0%|          |  0.00B / 1.20GB            

data/train-00000-of-00004.parquet: downloading bytes:           |  0.00B            

data/train-00001-of-00004.parquet: reconstructing file:   0%|          |  0.00B / 1.36GB            

data/train-00001-of-00004.parquet: downloading bytes:           |  0.00B            

data/train-00002-of-00004.parquet: reconstructing file:   0%|          |  0.00B / 2.15GB            

data/train-00002-of-00004.parquet: downloading bytes:           |  0.00B            

data/train-00003-of-00004.parquet: reconstructing file:   0%|          |  0.00B / 1.73GB            

data/train-00003-of-00004.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/8576 [00:00<?, ? examples/s]

Dataset({
    features: ['audio', 'text', 'gender'],
    num_rows: 8576
})
{'audio': Audio(sampling_rate=None, mono=True, decode=True, id=None), 'text': Value(dtype='string', id=None), 'gender': ClassLabel(names=['female', 'male'], id=None)}


In [ ]:
from getpass import getpass
from huggingface_hub import login
login(token=getpass("Paste HF token: "))

Paste HF token: ··········


## 6. Inspect one example

We look at **one** recording to learn the column names, the sampling rate and the language of the text.
`Audio(decode=False)` means: give us the raw audio file bytes (faster and avoids decoder problems).
Check the printed column names: this notebook expects `audio`, `text` and `gender`. If yours are different, change `TEXT_COL` / `GENDER_COL` in the next cells.

In [ ]:
AUDIO_COL, TEXT_COL, GENDER_COL = "audio", "text", "gender"
ds_raw = ds.cast_column(AUDIO_COL, Audio(decode=False))

ex = ds_raw[0]
print({k: v for k, v in ex.items() if k != AUDIO_COL})
a = ex[AUDIO_COL]
print("audio keys:", list(a.keys()), "| bytes present:", a.get("bytes") is not None)
info = sf.info(io.BytesIO(a["bytes"]))
print("sampling rate:", info.samplerate, "| channels:", info.channels, "| seconds:", round(info.duration, 2))

# turn the gender number (0/1) into a word (female/male)
feat = ds.features[GENDER_COL]
gender_of = (lambda g: feat.int2str(g)) if hasattr(feat, "int2str") else (lambda g: str(g))
print("gender of first example:", gender_of(ex[GENDER_COL]))

{'text': 'నరకవాసి.', 'gender': 0}
audio keys: ['bytes', 'path'] | bytes present: True
sampling rate: 16000 | channels: 1 | seconds: 2.16
gender of first example: female


## 7. Dataset statistics

We measure how long every recording is (reading only the file header, so it is quick), then print total hours, how many clips fall in different length ranges, and hours per speaker.

In [ ]:
from tqdm.auto import tqdm
durs, genders, texts = [], [], []
for ex in tqdm(ds_raw):
    durs.append(sf.info(io.BytesIO(ex[AUDIO_COL]["bytes"])).duration)
    genders.append(gender_of(ex[GENDER_COL]))
    texts.append(ex[TEXT_COL])
durs = np.array(durs)
genders = np.array(genders)

print("utterances:", len(durs), "| total hours:", round(durs.sum()/3600, 2))
for lo, hi in [(2, 15), (1, 20), (1, 30)]:
    m = (durs >= lo) & (durs <= hi)
    print(f"{lo}-{hi}s: {m.sum()} utts, {durs[m].sum()/3600:.2f} h")
for g in sorted(set(genders)):
    print(g, "hours:", round(durs[genders == g].sum()/3600, 2))

  0%|          | 0/8576 [00:00<?, ?it/s]

utterances: 8576 | total hours: 36.71
2-15s: 5216 utts, 11.82 h
1-20s: 6328 utts, 16.64 h
1-30s: 7552 utts, 24.95 h
female hours: 23.92
male hours: 12.79


## 8. Settings and Telugu text cleaning

Computers get confused by messy text. Our rules:
- Make the Unicode form standard (NFC) so the same letter is always stored the same way.
- Remove punctuation and any non-Telugu symbol.
- **Skip** training sentences that contain digits or English letters (so the model only learns clean Telugu spelling). Numbers in test sentences will be turned into Telugu words later, at inference time.

Only Telugu letters (Unicode range `0C00-0C7F`) and spaces stay.

**Settings:** `SEED` makes the random choice repeatable. `SR` is the audio quality we train with (22.05 kHz). We keep clips between 2 and 15 seconds.

In [ ]:
SEED, SR = 42, 22050
TRAIN_SEC, VAL_SEC = 5*3600, 1*3600
MIN_DUR, MAX_DUR = 2.0, 15.0
OUT = "/content/raw_data/Telugu"
os.makedirs(OUT, exist_ok=True)
random.seed(SEED)

NON_TELUGU = re.compile(r"[^\u0C00-\u0C7F ]")
HAS_DIGIT  = re.compile(r"[0-9\u0C66-\u0C6F]")      # English or Telugu digits
HAS_LATIN  = re.compile(r"[A-Za-z]")

def clean_text(t):
    t = unicodedata.normalize("NFC", str(t))
    t = t.replace("\u200c", "").replace("\u200d", "")   # invisible joiner characters
    t = NON_TELUGU.sub(" ", t)                           # remove punctuation, symbols, other scripts
    return re.sub(r"\s+", " ", t).strip()

def is_usable_text(raw):
    return not (HAS_DIGIT.search(raw) or HAS_LATIN.search(raw))

print(clean_text("నమస్కారం, మీరు ఎలా ఉన్నారు?"))
print("example sentence from the dataset:", texts[0], "->", clean_text(texts[0]))

నమస్కారం మీరు ఎలా ఉన్నారు
example sentence from the dataset: నరకవాసి. -> నరకవాసి


## 9. Pick ~5 h for training and ~1 h for validation



**Our selection strategy :**
1. Keep only usable clips: clean Telugu text without digits or English letters, and length 2 to 15 s.
2. Split the time budget **equally between the two speakers** (female and male), so both voices are learned: 2.5 h + 2.5 h for training, 0.5 h + 0.5 h for validation.
3. Shuffle with a fixed random seed (42) so the result can be repeated, and take clips until each speaker's budget is filled.
4. Validation clips are chosen only from clips **not used in training**, and their sentence text must not appear in the training set either.

Then each chosen file is converted to 22.05 kHz mono, loudness-normalised, and saved as `.wav` together with a `.lab` text file holding the clean sentence.

In [ ]:
# 1) usable clips
rows = []
for i in range(len(texts)):
    if not is_usable_text(str(texts[i])):
        continue
    text = clean_text(texts[i])
    if text and MIN_DUR <= durs[i] <= MAX_DUR:
        rows.append(dict(idx=i, text=text, dur=float(durs[i]), spk=genders[i].capitalize()))
speakers = sorted({r["spk"] for r in rows})
print("usable:", len(rows), "utts,", round(sum(r["dur"] for r in rows)/3600, 2), "h | speakers:", speakers)

# 2) equal time budget per speaker, seeded shuffle, no overlap
train_budget, val_budget = TRAIN_SEC / len(speakers), VAL_SEC / len(speakers)
random.shuffle(rows)
used = {s: [0.0, 0.0] for s in speakers}        # [train seconds, val seconds]
train, val, train_texts, seen_val = [], [], set(), set()
for r in rows:
    if used[r["spk"]][0] >= train_budget: continue
    train.append(r); used[r["spk"]][0] += r["dur"]; train_texts.add(r["text"])
train_ids = {r["idx"] for r in train}
for r in rows:
    s = r["spk"]
    if used[s][1] >= val_budget: continue
    if r["idx"] in train_ids or r["text"] in train_texts or r["text"] in seen_val: continue
    val.append(r); used[s][1] += r["dur"]; seen_val.add(r["text"])
print(f"train: {sum(r['dur'] for r in train)/3600:.2f} h ({len(train)}) | val: {sum(r['dur'] for r in val)/3600:.2f} h ({len(val)})")
for s in speakers:
    print(f"  {s}: train {used[s][0]/3600:.2f} h, val {used[s][1]/3600:.2f} h")

# 3) write wav + lab files
manifest = {"train": [], "val": []}
for name, part in [("train", train), ("val", val)]:
    for k, r in enumerate(part):
        ex = ds_raw[r["idx"]]
        y, sr0 = sf.read(io.BytesIO(ex[AUDIO_COL]["bytes"]), dtype="float32")
        if y.ndim > 1: y = y.mean(axis=1)
        y = librosa.resample(y, orig_sr=sr0, target_sr=SR)
        y = y / max(1e-9, np.abs(y).max()) * 0.95
        d = f"{OUT}/{r['spk']}"; os.makedirs(d, exist_ok=True)
        base = f"{name}_{r['spk']}_{k:05d}"
        sf.write(f"{d}/{base}.wav", y, SR)
        open(f"{d}/{base}.lab", "w", encoding="utf8").write(r["text"])
        manifest[name].append(dict(base=base, idx=r["idx"], spk=r["spk"], dur=round(r["dur"], 3)))
json.dump(manifest, open("/content/drive/MyDrive/fs2/indictts_telugu_split.json", "w"), ensure_ascii=False)
print("done")

# back up the prepared audio to Drive automatically (one zip)
!mkdir -p /content/drive/MyDrive/fs2/backup
!cd /content && zip -rq /content/drive/MyDrive/fs2/backup/raw_Telugu.zip raw_data/Telugu
!ls -lh /content/drive/MyDrive/fs2/backup/raw_Telugu.zip


usable: 5205 utts, 11.79 h | speakers: ['Female', 'Male']
train: 5.00 h (2110) | val: 1.00 h (423)
  Female: train 2.50 h, val 0.50 h
  Male: train 2.50 h, val 0.50 h
done
-rw------- 1 root root 716M Nov  6 09:02 /content/drive/MyDrive/fs2/backup/raw_Telugu.zip


## 10. Check the split

Two checks, both must give an **empty set** `set()`:
- no recording is used in both train and validation,
- no sentence is used in both train and validation.

In [ ]:
m = json.load(open("/content/drive/MyDrive/fs2/indictts_telugu_split.json"))
tr_idx = {e["idx"] for e in m["train"]}; va_idx = {e["idx"] for e in m["val"]}
read = lambda e: open(f"{OUT}/{e['spk']}/{e['base']}.lab", encoding="utf8").read()
tr_txt = {read(e) for e in m["train"]}; va_txt = {read(e) for e in m["val"]}
print("train:", len(m["train"]), "val:", len(m["val"]))
print("audio overlap   :", tr_idx & va_idx)
print("sentence overlap:", tr_txt & va_txt)
print("train hours:", round(sum(e["dur"] for e in m["train"])/3600, 2), "| val hours:", round(sum(e["dur"] for e in m["val"])/3600, 2))
!ls /content/raw_data/Telugu

train: 2110 val: 423
audio overlap   : set()
sentence overlap: set()
train hours: 5.0 | val hours: 1.0
Female	Male


## 11. Back up the prepared data to Drive

Many small files are very slow on Drive, so we pack everything into **one zip** and copy that.

In [ ]:
!cd /content && zip -rq /content/drive/MyDrive/fs2/backup/raw_Telugu.zip raw_data/Telugu
!ls -lh /content/drive/MyDrive/fs2/backup/raw_Telugu.zip

-rw------- 1 root root 716M Nov  6 09:08 /content/drive/MyDrive/fs2/backup/raw_Telugu.zip


## 12. Install Montreal Forced Aligner (MFA)

**What does MFA do?** The model needs to know *when* each sound happens inside a recording (for example, "ka" is from 0.30 s to 0.42 s). MFA listens to the audio, reads the text, and writes these timings to `.TextGrid` files.



In [ ]:
!pip install -q condacolab
import condacolab
condacolab.install()


📢 Announcement 📢
condacolab==0.2 will be released soon! Try it with:

    !pip install -q https://github.com/conda-incubator/condacolab/archive/main.zip
    import condacolab
    condacolab.install()

0.2.x introduces a new installation method based on Pixi, with customizable Python versions.
This may be breaking for your workflow. If that's the case, please report it at
https://github.com/conda-incubator/condacolab and pin your `pip install` command to:

    !pip install -q 'condacolab==0.1.*'

⏬ Downloading https://github.com/conda-forge/miniforge/releases/download/26.3.2-3/Miniforge3-26.3.2-3-Linux-x86_64.sh...
📦 Installing...
📌 Adjusting configuration...
🩹 Patching environment...
⏲ Done in 0:00:15
🔁 Restarting kernel...


## 13. Install MFA

In [ ]:
!mamba install -q -y -c conda-forge montreal-forced-aligner
!mfa version

3.4.3


## 14. Build a Telugu pronunciation dictionary

MFA needs a dictionary: *word → sounds*. For English we had a ready one. For Telugu we do not have a ready dictionary here, but Telugu is written almost exactly as it is spoken. So we use each **Telugu character as one "sound"**. This is called a *grapheme dictionary*.
Example: `కాలం  →  క ా ల ం`

In [ ]:
import glob, collections
cnt = collections.Counter(w for f in glob.glob("/content/raw_data/Telugu/*/*.lab")
                          for w in open(f, encoding="utf8").read().split())
print("different words:", len(cnt), "| total words:", sum(cnt.values()))
with open("/content/telugu_dict.txt", "w", encoding="utf8") as f:
    for w in sorted(cnt):
        f.write(w + "\t" + " ".join(list(w)) + "\n")
!head -5 /content/telugu_dict.txt
!mkdir -p /content/drive/MyDrive/fs2/backup
!cp /content/telugu_dict.txt /content/drive/MyDrive/fs2/backup/


different words: 12545 | total words: 30069
ం	ం
అంగన్	అ ం గ న ్
అంగరక్షకులలో	అ ం గ ర క ్ ష క ు ల ల ో
అంగీకరించాడు	అ ం గ ీ క ర ి ం చ ా డ ు
అంగీకరించారు	అ ం గ ీ క ర ి ం చ ా ర ు


## 15. Train the aligner on our own data and create TextGrids



In [ ]:
import subprocess, time, os, shutil

BK = "/content/drive/MyDrive/fs2/backup"
DRIVE_TG = "/content/drive/MyDrive/fs2/preprocessed_data/Telugu/TextGrid"
LOCAL_LOG = "/content/mfa_train.log"
os.makedirs(BK, exist_ok=True)

def mfa_running():
    # [m] trick: stops pgrep from matching its own command line
    return subprocess.run("pgrep -f '[m]fa train'", shell=True, capture_output=True).stdout.strip() != b""

# 1) start MFA in the background (skipped if it is already running or already finished)
if not mfa_running() and not os.path.exists("/content/telugu_acoustic.zip"):
    cmd = ("mfa train --clean -j 2 --output_directory /content/telugu_textgrid "
           "/content/raw_data/Telugu /content/telugu_dict.txt /content/telugu_acoustic.zip")
    subprocess.Popen(f"nohup {cmd} > {LOCAL_LOG} 2>&1 &", shell=True)
    print("MFA started in the background")

# 2) while it runs: every 2 minutes copy the log to Drive and show the latest progress
while mfa_running():
    time.sleep(120)
    if os.path.exists(LOCAL_LOG):
        shutil.copy2(LOCAL_LOG, f"{BK}/mfa_train.log")
        print(time.strftime("%H:%M:%S"), "|", open(LOCAL_LOG, errors="ignore").read()[-200:].replace("\n", " "))

# 3) when it finishes: save everything to Drive automatically
if os.path.exists(LOCAL_LOG):
    shutil.copy2(LOCAL_LOG, f"{BK}/mfa_train.log")
if os.path.isdir("/content/telugu_textgrid"):
    shutil.copytree("/content/telugu_textgrid", DRIVE_TG, dirs_exist_ok=True)
for f in ["telugu_acoustic.zip", "telugu_dict.txt"]:
    if os.path.exists(f"/content/{f}"):
        shutil.copy2(f"/content/{f}", f"{BK}/{f}")
n_local = sum(len(fs) for _, _, fs in os.walk("/content/telugu_textgrid")) if os.path.isdir("/content/telugu_textgrid") else 0
n_drive = sum(len(fs) for _, _, fs in os.walk(DRIVE_TG)) if os.path.isdir(DRIVE_TG) else 0
print("TextGrid files local:", n_local, "| saved on Drive:", n_drive)
print("model on Drive:", os.path.exists(f"{BK}/telugu_acoustic.zip"))

MFA started in the background
09:13:44 |                                        INFO     monophone - Iteration 1 of 40                                           INFO     Generating alignments...                                               
09:15:44 |                                        INFO     monophone - Iteration 1 of 40                                           INFO     Generating alignments...                                               
09:17:44 |                                        INFO     monophone - Iteration 1 of 40                                           INFO     Generating alignments...                                               
09:19:44 |                                        100% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2,533/2,533  [ 0:01:29 < 0:00:00 , 27 it/s ]  INFO     Accumulating statistics...                                             
09:21:44 | 533  [ 0:00:13 < 0:00:01 , 121 it/s ]  INFO     monophone - Iteration 4 of 40                                  

In [ ]:
import os, shutil
from google.colab import drive

print("Drive mounted?", os.path.ismount("/content/drive"))
print("what is inside now:", os.listdir("/content/drive") if os.path.exists("/content/drive") else "nothing")

if os.path.ismount("/content/drive"):
    print("Drive is already mounted - nothing to do")
else:
    # not a real mount, so this only removes the local junk folder, never your real Drive
    shutil.rmtree("/content/drive", ignore_errors=True)
    drive.mount("/content/drive")

# check that your backups are really there
B = "/content/drive/MyDrive/fs2"
for p in ["backup/raw_Telugu.zip", "backup/telugu_acoustic.zip", "backup/telugu_dict.txt",
          "preprocessed_data/Telugu/TextGrid", "generator_universal.pth.tar.zip", "indictts_telugu_split.json"]:
    print("FOUND  " if os.path.exists(f"{B}/{p}") else "MISSING", p)

Drive mounted? False
what is inside now: ['MyDrive']
Mounted at /content/drive
FOUND   backup/raw_Telugu.zip
MISSING backup/telugu_acoustic.zip
FOUND   backup/telugu_dict.txt
MISSING preprocessed_data/Telugu/TextGrid
FOUND   generator_universal.pth.tar.zip
FOUND   indictts_telugu_split.json


In [ ]:
import os, subprocess, shutil
running = subprocess.run("pgrep -f '[m]fa train'", shell=True, capture_output=True).stdout.strip() != b""
n = sum(len(f) for _, _, f in os.walk("/content/telugu_textgrid")) if os.path.isdir("/content/telugu_textgrid") else 0
has_model = os.path.exists("/content/telugu_acoustic.zip")
print("MFA still running:", running)
print("local TextGrid files:", n, "(expected 2533) | local model:", has_model)
if os.path.exists("/content/mfa_train.log"):
    print("log tail:", open("/content/mfa_train.log", errors="ignore").read()[-300:].replace("\n", " "))

if os.path.ismount("/content/drive") and not running and n > 0 and has_model:
    B = "/content/drive/MyDrive/fs2"
    shutil.copytree("/content/telugu_textgrid", f"{B}/preprocessed_data/Telugu/TextGrid", dirs_exist_ok=True)
    for f in ["telugu_acoustic.zip", "telugu_dict.txt", "mfa_train.log"]:
        if os.path.exists(f"/content/{f}"):
            shutil.copy2(f"/content/{f}", f"{B}/backup/{f}")
    print("SAVED to Drive. Continue with cell 18b.")
elif running:
    print("MFA is still running - wait for it, then run this cell again.")
else:
    print("No finished MFA output on this machine - do Step 2.")

MFA still running: False
local TextGrid files: 0 (expected 2533) | local model: False
No finished MFA output on this machine - do Step 2.


In [ ]:
import os
from google.colab import drive
if not os.path.ismount("/content/drive"):
    drive.mount('/content/drive')
assert os.path.ismount("/content/drive"), "Drive is not mounted"
B = "/content/drive/MyDrive/fs2"
for p in ["backup/raw_Telugu.zip", "indictts_telugu_split.json", "backup/telugu_dict.txt"]:
    print("FOUND  " if os.path.exists(f"{B}/{p}") else "MISSING", p)
!test -d /content/raw_data/Telugu || unzip -q -o /content/drive/MyDrive/fs2/backup/raw_Telugu.zip -d /content/
n = sum(f.endswith(".wav") for _, _, fs in os.walk("/content/raw_data/Telugu") for f in fs)
print("wav files restored:", n, "(expected 2533)")

FOUND   backup/raw_Telugu.zip
FOUND   indictts_telugu_split.json
FOUND   backup/telugu_dict.txt
wav files restored: 2533 (expected 2533)


In [ ]:
!pip install -q condacolab
import condacolab
condacolab.install()


📢 Announcement 📢
condacolab==0.2 will be released soon! Try it with:

    !pip install -q https://github.com/conda-incubator/condacolab/archive/main.zip
    import condacolab
    condacolab.install()

0.2.x introduces a new installation method based on Pixi, with customizable Python versions.
This may be breaking for your workflow. If that's the case, please report it at
https://github.com/conda-incubator/condacolab and pin your `pip install` command to:

    !pip install -q 'condacolab==0.1.*'

⏬ Downloading https://github.com/conda-forge/miniforge/releases/download/26.3.2-3/Miniforge3-26.3.2-3-Linux-x86_64.sh...
📦 Installing...
📌 Adjusting configuration...
🩹 Patching environment...
⏲ Done in 0:00:15
🔁 Restarting kernel...


In [ ]:
!mamba install -q -y -c conda-forge montreal-forced-aligner
!mfa version

warning  libmamba [libhwloc-2.13.0-default_he001693_1000] The following files were already present in the environment:
    - lib/libhwloc.so
    - lib/libhwloc.so.15
    - lib/libhwloc.so.15.10.2
warning  libmamba [tbb-2023.1.0-hfd44327_0] The following files were already present in the environment:
    - lib/libtbb.so.12
    - lib/libtbbbind_2_5.so.3
    - lib/libtbbmalloc.so.2
    - lib/libtbbmalloc_proxy.so.2
warning  libmamba [numpy-2.5.4-py313hb5f73ae_0] The following files were already present in the environment:
    - bin/f2py
    - bin/numpy-config
warning  libmamba [numba-0.68.1-py313h901f96d_0] The following files were already present in the environment:
    - bin/numba
warning  libmamba [pygments-2.21.0-pyhcf101f3_0] The following files were already present in the environment:
    - bin/pygmentize
warning  libmamba [markdown-it-py-4.2.0-pyhd8ed1ab_0] The following files were already present in the environment:
    - bin/markdown-it
warning  libmamba [httpx-0.28.1-pyhd8ed1ab_

In [ ]:
import os, glob, collections
from google.colab import drive
if not os.path.ismount("/content/drive"):
    drive.mount('/content/drive')
assert os.path.ismount("/content/drive"), "Drive is not mounted"

cnt = collections.Counter(w for f in glob.glob("/content/raw_data/Telugu/*/*.lab")
                          for w in open(f, encoding="utf8").read().split())
print("different words:", len(cnt), "| total words:", sum(cnt.values()))
with open("/content/telugu_dict.txt", "w", encoding="utf8") as f:
    for w in sorted(cnt):
        f.write(w + "\t" + " ".join(list(w)) + "\n")
!head -5 /content/telugu_dict.txt
!mkdir -p /content/drive/MyDrive/fs2/backup
!cp /content/telugu_dict.txt /content/drive/MyDrive/fs2/backup/

different words: 12545 | total words: 30069
ం	ం
అంగన్	అ ం గ న ్
అంగరక్షకులలో	అ ం గ ర క ్ ష క ు ల ల ో
అంగీకరించాడు	అ ం గ ీ క ర ి ం చ ా డ ు
అంగీకరించారు	అ ం గ ీ క ర ి ం చ ా ర ు


In [ ]:
import subprocess, time, os, shutil

BK = "/content/drive/MyDrive/fs2/backup"
DRIVE_TG = "/content/drive/MyDrive/fs2/preprocessed_data/Telugu/TextGrid"
LOCAL_LOG = "/content/mfa_train.log"
assert os.path.ismount("/content/drive"), "Google Drive is NOT mounted - run drive.mount first (do not continue, files would be lost)"
os.makedirs(BK, exist_ok=True)

def mfa_running():
    # [m] trick: stops pgrep from matching its own command line
    return subprocess.run("pgrep -f '[m]fa train'", shell=True, capture_output=True).stdout.strip() != b""

# shorter training plan: monophone + triphone only (no LDA / speaker-adaptation stages)
MFA_CFG = """training:
  - monophone:
      num_iterations: 20
      max_gaussians: 1000
      boost_silence: 1.25
  - triphone:
      num_iterations: 20
      num_leaves: 2000
      max_gaussians: 10000
      cluster_threshold: -1
      boost_silence: 1.25
      power: 0.25
"""
open("/content/mfa_config.yaml", "w").write(MFA_CFG)

# 1) start MFA in the background (skipped if it is already running or already finished)
if not mfa_running() and not os.path.exists("/content/telugu_acoustic.zip"):
    cmd = ("mfa train --clean -j 2 --config_path /content/mfa_config.yaml --output_directory /content/telugu_textgrid "
           "/content/raw_data/Telugu /content/telugu_dict.txt /content/telugu_acoustic.zip")
    subprocess.Popen(f"nohup {cmd} > {LOCAL_LOG} 2>&1 &", shell=True)
    print("MFA started in the background")

# 2) while it runs: every 2 minutes copy the log to Drive and show the latest progress
while mfa_running():
    time.sleep(120)
    if os.path.exists(LOCAL_LOG):
        shutil.copy2(LOCAL_LOG, f"{BK}/mfa_train.log")
        print(time.strftime("%H:%M:%S"), "|", open(LOCAL_LOG, errors="ignore").read()[-200:].replace("\n", " "))

# 3) when it finishes: save everything to Drive automatically
if os.path.exists(LOCAL_LOG):
    shutil.copy2(LOCAL_LOG, f"{BK}/mfa_train.log")
if os.path.isdir("/content/telugu_textgrid"):
    shutil.copytree("/content/telugu_textgrid", DRIVE_TG, dirs_exist_ok=True)
for f in ["telugu_acoustic.zip", "telugu_dict.txt"]:
    if os.path.exists(f"/content/{f}"):
        shutil.copy2(f"/content/{f}", f"{BK}/{f}")
if not os.path.exists("/content/telugu_acoustic.zip"):
    print("MFA did NOT finish. Last lines of the log:")
    print(open(LOCAL_LOG, errors="ignore").read()[-1500:])
n_local = sum(len(fs) for _, _, fs in os.walk("/content/telugu_textgrid")) if os.path.isdir("/content/telugu_textgrid") else 0
n_drive = sum(len(fs) for _, _, fs in os.walk(DRIVE_TG)) if os.path.isdir(DRIVE_TG) else 0
print("TextGrid files local:", n_local, "| saved on Drive:", n_drive)
print("model on Drive:", os.path.exists(f"{BK}/telugu_acoustic.zip"))

MFA started in the background
10:38:31 |                                        INFO     monophone - Iteration 1 of 20                                           INFO     Generating alignments...                                               
10:40:31 |                                        INFO     monophone - Iteration 1 of 20                                           INFO     Generating alignments...                                               
10:42:31 | 2,000  [ 0:00:06 < -:--:-- , ? it/s ]  INFO     monophone - Iteration 3 of 20                                           INFO     Generating alignments...                                               
10:44:31 | 2,000  [ 0:00:09 < -:--:-- , ? it/s ]  INFO     monophone - Iteration 7 of 20                                           INFO     Generating alignments...                                               
10:46:31 | 2,000  [ 0:00:07 < -:--:-- , ? it/s ]  INFO     monophone - Iteration 14 of 20                                 

## 16. Save the alignment results to Drive

In [ ]:
!mkdir -p /content/drive/MyDrive/fs2/preprocessed_data/Telugu
!cp -r /content/telugu_textgrid /content/drive/MyDrive/fs2/preprocessed_data/Telugu/TextGrid
!cp /content/telugu_dict.txt /content/telugu_acoustic.zip /content/drive/MyDrive/fs2/backup/
!find /content/drive/MyDrive/fs2/preprocessed_data/Telugu/TextGrid -name "*.TextGrid" | wc -l

5020


## 17. Check the repo's Telugu folder (preflight)



In [ ]:
import os, shutil
T = "/content/project/Telugu"
need = ["train.py", "preprocess.py", "inference.py", "config.yaml", "dataset.py", "plot_losses.py",
        "preprocessor/preprocessor.py", "text/symbols.py", "utils/model.py", "utils/tools.py"]
if not os.path.exists(f"{T}/plot_losses.py") and os.path.exists("/content/project/English/plot_losses.py"):
    shutil.copy2("/content/project/English/plot_losses.py", f"{T}/plot_losses.py")
    print("copied plot_losses.py from English")
for n in need:
    print("OK     " if os.path.exists(f"{T}/{n}") else "MISSING", n)
print("---- text/symbols.py ----")
print(open(f"{T}/text/symbols.py", encoding="utf8").read()[:1500])
print("---- Telugu test sentences in inference.py? ----")
src = open(f"{T}/inference.py", encoding="utf8").read()
for sid in ["T1", "T2a", "T2b", "T3", "T4"]:
    print(sid, "found" if sid in src else "NOT FOUND")

MISSING train.py
MISSING preprocess.py
MISSING inference.py
MISSING config.yaml
MISSING dataset.py
MISSING plot_losses.py
MISSING preprocessor/preprocessor.py
MISSING text/symbols.py
MISSING utils/model.py
MISSING utils/tools.py
---- text/symbols.py ----


FileNotFoundError: [Errno 2] No such file or directory: '/content/project/Telugu/text/symbols.py'

In [ ]:
import os, shutil
P, E, T = "/content/project", "/content/project/English", "/content/project/Telugu"
print("project folder exists:", os.path.isdir(P))
if not os.path.isdir(P):
    print("-> Run cell 18b first (it downloads the project), then run this cell again.")
else:
    print("project contains:", sorted(os.listdir(P)))
    print("English contains:", sorted(os.listdir(E)) if os.path.isdir(E) else "NO English folder")
    print("Telugu contains :", sorted(os.listdir(T)) if os.path.isdir(T) else "NO Telugu folder")
    if os.path.isdir(E) and not os.path.exists(f"{T}/train.py"):
        os.makedirs(T, exist_ok=True)
        skip = shutil.ignore_patterns("__pycache__", "*.pth.tar", "*.zip", "checkpoints", ".ipynb_checkpoints")
        for name in os.listdir(E):
            if name in ("checkpoints", "__pycache__"):
                continue
            s, d = f"{E}/{name}", f"{T}/{name}"
            if os.path.isdir(s):
                shutil.copytree(s, d, dirs_exist_ok=True, ignore=skip)
            elif not os.path.exists(d):
                shutil.copy2(s, d)
        print("Copied English code into Telugu.")
        print("Telugu now contains:", sorted(os.listdir(T)))

project folder exists: False
-> Run cell 18b first (it downloads the project), then run this cell again.


In [ ]:
import os
B = "/content/drive/MyDrive/fs2"
tg = f"{B}/preprocessed_data/Telugu/TextGrid"
n = sum(len(f) for _, _, f in os.walk(tg)) if os.path.isdir(tg) else 0
print("TextGrids on Drive:", n, "(expected 2533)")
print("model on Drive:", os.path.exists(f"{B}/backup/telugu_acoustic.zip"))

TextGrids on Drive: 5020 (expected 2533)
model on Drive: True


In [ ]:
import os, collections

def inspect(tg, label):
    print("=====", label, tg)
    if not os.path.isdir(tg):
        print("folder not found"); return
    ext, per_folder = collections.Counter(), collections.Counter()
    for root, _, fs in os.walk(tg):
        for f in fs:
            ext[os.path.splitext(f)[1]] += 1
            if f.endswith(".TextGrid"):
                per_folder[os.path.relpath(root, tg)] += 1
    print("top level   :", os.listdir(tg))
    print("by extension:", dict(ext))
    print("TextGrids per folder:", dict(per_folder), "| total:", sum(per_folder.values()))

inspect("/content/drive/MyDrive/fs2/preprocessed_data/Telugu/TextGrid", "DRIVE")
inspect("/content/telugu_textgrid", "LOCAL (this session)")
raw = collections.Counter(os.path.basename(r) for r, _, fs in os.walk("/content/raw_data/Telugu") for f in fs if f.endswith(".wav"))
print("wav files per speaker folder:", dict(raw), "| total:", sum(raw.values()))

===== DRIVE /content/drive/MyDrive/fs2/preprocessed_data/Telugu/TextGrid
top level   : ['Female', 'Male', 'telugu_textgrid']
by extension: {'.TextGrid': 5020}
TextGrids per folder: {'Female': 1158, 'Male': 1352, 'telugu_textgrid/Male': 1352, 'telugu_textgrid/Female': 1158} | total: 5020
===== LOCAL (this session) /content/telugu_textgrid
top level   : ['Female', 'Male']
by extension: {'.TextGrid': 2510}
TextGrids per folder: {'Female': 1158, 'Male': 1352} | total: 2510
wav files per speaker folder: {'Female': 1180, 'Male': 1353} | total: 2533


In [ ]:
import os, shutil, glob
TG = "/content/drive/MyDrive/fs2/preprocessed_data/Telugu/TextGrid"
nested = f"{TG}/telugu_textgrid"
cnt = lambda d: len(glob.glob(f"{d}/*.TextGrid"))
print("top-level Female:", cnt(f"{TG}/Female"), "Male:", cnt(f"{TG}/Male"))
if os.path.isdir(nested):
    if cnt(f"{TG}/Female") >= 1158 and cnt(f"{TG}/Male") >= 1352:
        shutil.rmtree(nested)
        print("removed the duplicate folder")
    else:
        print("top-level counts too low - NOT removing")
print("top level now:", os.listdir(TG))

# which recordings have no TextGrid?
raw = "/content/raw_data/Telugu"
missing = []
for spk in ["Female", "Male"]:
    for w in glob.glob(f"{raw}/{spk}/*.wav"):
        b = os.path.splitext(os.path.basename(w))[0]
        if not os.path.exists(f"{TG}/{spk}/{b}.TextGrid"):
            missing.append(b)
print("no TextGrid:", len(missing),
      "| train:", sum(m.startswith("train_") for m in missing),
      "| val:", sum(m.startswith("val_") for m in missing))
open("/content/drive/MyDrive/fs2/backup/unaligned_utterances.txt", "w").write("\n".join(missing))

top-level Female: 1158 Male: 1352
removed the duplicate folder
top level now: ['Female', 'Male']
no TextGrid: 23 | train: 20 | val: 3


428

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
%%bash
set -e
cd /content
if [ -d project ]; then cd project && git pull; else git clone https://github.com/Narmadas-31/fastspeech2-english-telugu.git project; fi
pip install -q librosa soundfile pyyaml tgt pyworld g2p_en inflect unidecode pypinyin scikit-learn tensorboard matplotlib pandas scipy
mkdir -p /content/project/Telugu/hifigan
cp /content/drive/MyDrive/fs2/generator_universal.pth.tar.zip /content/project/Telugu/hifigan/
cd /content/project/Telugu/hifigan && unzip -oq generator_universal.pth.tar.zip
python -c "import torch; print(torch.__version__, torch.cuda.is_available())"

2.11.0+cpu False


Cloning into 'project'...


In [ ]:
import os, shutil
P, E, T = "/content/project", "/content/project/English", "/content/project/Telugu"
print("English contains:", sorted(os.listdir(E)) if os.path.isdir(E) else "NO English folder")
print("Telugu contains :", sorted(os.listdir(T)) if os.path.isdir(T) else "NO Telugu folder")
if os.path.isdir(E) and not os.path.exists(f"{T}/train.py"):
    os.makedirs(T, exist_ok=True)
    skip = shutil.ignore_patterns("__pycache__", "*.pth.tar", "*.zip", "checkpoints", ".ipynb_checkpoints")
    for name in os.listdir(E):
        if name in ("checkpoints", "__pycache__"):
            continue
        s, d = f"{E}/{name}", f"{T}/{name}"
        if os.path.isdir(s):
            shutil.copytree(s, d, dirs_exist_ok=True, ignore=skip)
        elif not os.path.exists(d):
            shutil.copy2(s, d)
    print("Copied English code into Telugu.")
    print("Telugu now contains:", sorted(os.listdir(T)))

English contains: ['apply_patches.py', 'audio', 'checkpoints', 'config.yaml', 'config_ref', 'dataset.py', 'evaluate.py', 'hifigan', 'inference.py', 'lexicon', 'make_config.py', 'model', 'plot_losses.py', 'preprocess.py', 'preprocessor', 'requirements_ref.txt', 'synthesize.py', 'text', 'train.py', 'transformer', 'utils']
Telugu contains : ['audio', 'checkpoints', 'config.yaml', 'config_ref', 'dataset.py', 'evaluate.py', 'hifigan', 'inference.py', 'lexicon', 'model', 'preprocess.py', 'preprocessor', 'requirements_ref.txt', 'synthesize.py', 'text', 'train.py', 'transformer', 'utils']


## 18. Set up the environment for preprocessing



In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%%bash
set -e
cd /content
if [ -d project ]; then cd project && git pull; else git clone https://github.com/Narmadas-31/fastspeech2-english-telugu.git project; fi
pip install -q librosa soundfile pyyaml tgt pyworld g2p_en inflect unidecode pypinyin scikit-learn tensorboard matplotlib pandas scipy
mkdir -p /content/project/Telugu/hifigan
cp /content/drive/MyDrive/fs2/generator_universal.pth.tar.zip /content/project/Telugu/hifigan/
cd /content/project/Telugu/hifigan && unzip -oq generator_universal.pth.tar.zip
python -c "import torch; print(torch.__version__, torch.cuda.is_available())"

## 19. Copy data to the local Colab disk


In [ ]:
%%bash
mkdir -p /content/raw_data /content/preprocessed_data/Telugu
if [ ! -d /content/raw_data/Telugu ]; then unzip -q -o /content/drive/MyDrive/fs2/backup/raw_Telugu.zip -d /content/; fi
cp -r /content/drive/MyDrive/fs2/preprocessed_data/Telugu/TextGrid /content/preprocessed_data/Telugu/
echo "wav files:" $(find /content/raw_data/Telugu -name "*.wav" | wc -l)
echo "TextGrid files:" $(find /content/preprocessed_data/Telugu/TextGrid -name "*.TextGrid" | wc -l)

## 20. Create two helper scripts (config and symbols)



In [ ]:
%%writefile /content/drive/MyDrive/fs2/patch_telugu_config.py
import yaml
CFG = "/content/project/Telugu/config.yaml"
D = "/content/drive/MyDrive/fs2/output/Telugu"
cfg = yaml.safe_load(open(CFG, encoding="utf8"))
pp = cfg["preprocessing"]
pp.setdefault("path", {})
pp["path"].update(corpus_path="/content/raw_data/Telugu", raw_path="/content/raw_data/Telugu",
                  preprocessed_path="/content/preprocessed_data/Telugu")
cfg["train"]["path"] = {"ckpt_path": f"{D}/ckpt", "log_path": f"{D}/log", "result_path": f"{D}/result"}
cfg["train"]["step"]["total_step"] = 10000
yaml.safe_dump(cfg, open(CFG, "w", encoding="utf8"), sort_keys=False, allow_unicode=True)
print(yaml.safe_dump(cfg, allow_unicode=True, sort_keys=False))

In [ ]:
%%writefile /content/drive/MyDrive/fs2/patch_telugu_symbols.py
import glob, json, os
import tgt
T = "/content/project/Telugu"
DRIVE_JSON = "/content/drive/MyDrive/fs2/telugu_phones.json"
files = glob.glob("/content/preprocessed_data/Telugu/TextGrid/*/*.TextGrid")
if files:
    ph = set()
    for f in files:
        tier = tgt.io.read_textgrid(f).get_tier_by_name("phones")
        for it in (getattr(tier, "intervals", None) or tier._objects):
            if it.text.strip():
                ph.add(it.text.strip())
    ph -= {"sil", "sp", "spn"}
    phones = sorted(ph)
    json.dump(phones, open(DRIVE_JSON, "w", encoding="utf8"), ensure_ascii=False)
else:
    phones = json.load(open(DRIVE_JSON, encoding="utf8"))
json.dump(phones, open(f"{T}/text/telugu_phones.json", "w", encoding="utf8"), ensure_ascii=False)

path = f"{T}/text/symbols.py"
sy = open(path, encoding="utf8").read()
if "TELUGU_PHONES" not in sy:
    sy += """

# --- TELUGU_PHONES: Telugu grapheme symbols produced by the MFA alignment ---
import json as _json, os as _os
try:
    _tp = _json.load(open(_os.path.join(_os.path.dirname(__file__), "telugu_phones.json"), encoding="utf8"))
except Exception:
    _tp = []
symbols = symbols + [s for s in ["@" + p for p in _tp] if s not in symbols]
"""
    open(path, "w", encoding="utf8").write(sy)
print(len(phones), "Telugu symbols:", " ".join(phones))

In [ ]:
!python /content/drive/MyDrive/fs2/patch_telugu_config.py
!python /content/drive/MyDrive/fs2/patch_telugu_symbols.py
!tail -12 /content/project/Telugu/text/symbols.py

## 21. Fix TextGrid compatibility issue

Patch the preprocessor to work with the newer `tgt` library (`tier.intervals`). It is safe to run this more than once.

In [ ]:
!sed -i 's/tier\._objects/tier.intervals/' /content/project/Telugu/preprocessor/preprocessor.py

## 22. Run preprocessing

Extract mel-spectrograms, pitch, energy and phoneme durations; create `train.txt`, `val.txt`, `stats.json` and `speakers.json`.

In [ ]:
%cd /content/project/Telugu
!mkdir -p /content/drive/MyDrive/fs2/logs
!python preprocess.py config.yaml 2>&1 | tee /content/drive/MyDrive/fs2/logs/telugu_preprocess.log

# save the preprocessed features to Drive automatically
!cd /content && zip -rq /content/drive/MyDrive/fs2/preprocessed_Telugu.zip preprocessed_data/Telugu -x "preprocessed_data/Telugu/TextGrid/*"
!ls -lh /content/drive/MyDrive/fs2/preprocessed_Telugu.zip


/content/project/Telugu
/usr/local/lib/python3.13/dist-packages/pyworld/__init__.py:13: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
Traceback (most recent call last):
  File "/content/project/Telugu/preprocess.py", line 14, in <module>
    preprocessor = Preprocessor(config)
  File "/content/project/Telugu/preprocessor/preprocessor.py", line 19, in __init__
    self.in_dir = config["path"]["raw_path"]
                  ~~~~~~^^^^^^^^
KeyError: 'path'
-rw------- 1 root root 200 Nov  7 00:38 /content/drive/MyDrive/fs2/preprocessed_Telugu.zip


In [ ]:
### Delete the empty zip:

!rm -f /content/drive/MyDrive/fs2/preprocessed_Telugu.zip

In [ ]:

#### Replace preprocess.py with a version that reads config.yaml correctly

%%writefile /content/project/Telugu/preprocess.py
import argparse

import yaml

from preprocessor.preprocessor import Preprocessor

if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("config", type=str, nargs="?", default="config.yaml",
                        help="path to config.yaml")
    args = parser.parse_args()

    cfg = yaml.load(open(args.config, "r", encoding="utf-8"), Loader=yaml.FullLoader)
    config = cfg["preprocessing"]

    # audio (.wav) and text (.lab) files are already prepared, so we go straight
    # to feature extraction from the wavs and TextGrids
    preprocessor = Preprocessor(config)
    preprocessor.build_from_path()

Overwriting /content/project/Telugu/preprocess.py


In [ ]:
%cd /content/project/Telugu
!python preprocess.py config.yaml 2>&1 | tee /content/drive/MyDrive/fs2/logs/telugu_preprocess.log

/content/project/Telugu
/usr/local/lib/python3.13/dist-packages/pyworld/__init__.py:13: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
Traceback (most recent call last):
  File "/content/project/Telugu/preprocess.py", line 18, in <module>
    preprocessor = Preprocessor(config)
  File "/content/project/Telugu/preprocessor/preprocessor.py", line 43, in __init__
    self.STFT = Audio.stft.TacotronSTFT(
                ~~~~~~~~~~~~~~~~~~~~~~~^
        config["preprocessing"]["stft"]["filter_length"],
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<5 lines>...
        config["preprocessing"]["mel"]["mel_fmax"],
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "/content/project/Telugu/audio/stft.py", line 144, in __init__
    self.stft_f

In [ ]:
import re, glob

changed = []
for path in glob.glob("/content/project/Telugu/audio/*.py"):
    src = open(path, encoding="utf-8").read()
    new = src

    # pad_center(data, size) -> pad_center(data, size=size)
    new = re.sub(r"pad_center\(\s*([^,()=]+?)\s*,\s*([^,()=]+?)\s*\)",
                 r"pad_center(\1, size=\2)", new)

    # librosa_mel_fn(sr, n_fft, n_mels, fmin, fmax) -> keyword arguments
    new = re.sub(
        r"librosa_mel_fn\(\s*([^,()=]+?)\s*,\s*([^,()=]+?)\s*,\s*([^,()=]+?)\s*,\s*([^,()=]+?)\s*,\s*([^,()=]+?)\s*\)",
        r"librosa_mel_fn(sr=\1, n_fft=\2, n_mels=\3, fmin=\4, fmax=\5)", new)

    if new != src:
        open(path, "w", encoding="utf-8").write(new)
        changed.append(path)

print("patched:", changed)
!grep -n "pad_center(\|librosa_mel_fn(" /content/project/Telugu/audio/*.py

patched: ['/content/project/Telugu/audio/stft.py', '/content/project/Telugu/audio/audio_processing.py']
/content/project/Telugu/audio/audio_processing.py:57:    win_sq = librosa_util.pad_center(win_sq, size=n_fft)
/content/project/Telugu/audio/stft.py:42:            fft_window = pad_center(fft_window, size=filter_length)
/content/project/Telugu/audio/stft.py:145:        mel_basis = librosa_mel_fn(sr=sampling_rate, n_fft=filter_length, n_mels=n_mel_channels, fmin=mel_fmin, fmax=mel_fmax)


In [ ]:
%cd /content/project/Telugu
!python preprocess.py config.yaml 2>&1 | tee /content/drive/MyDrive/fs2/logs/telugu_preprocess.log

/content/project/Telugu
/usr/local/lib/python3.13/dist-packages/pyworld/__init__.py:13: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
Processing Data ...
100%|██████████| 2/2 [05:13<00:00, 156.81s/it]
Computing statistic quantities ...
Total time: 4.165553439153439 hours


In [ ]:
import os
P = "/content/preprocessed_data/Telugu"
for d in ["mel", "pitch", "energy", "duration"]:
    print(d, len(os.listdir(f"{P}/{d}")) if os.path.isdir(f"{P}/{d}") else "MISSING")
for f in ["train.txt", "val.txt", "stats.json", "speakers.json"]:
    print(f, "OK" if os.path.exists(f"{P}/{f}") else "MISSING")

tr = [l for l in open(f"{P}/train.txt", encoding="utf8")] if os.path.exists(f"{P}/train.txt") else []
va = [l for l in open(f"{P}/val.txt", encoding="utf8")] if os.path.exists(f"{P}/val.txt") else []
print("train lines:", len(tr), "| val lines:", len(va))
print("lines starting with val_ inside train.txt:", sum(l.startswith("val_") for l in tr))
print("first train line:", tr[0].strip() if tr else None)
print(open(f"{P}/speakers.json").read())

mel 2510
pitch 2510
energy 2510
duration 2510
train.txt OK
val.txt OK
stats.json OK
speakers.json OK
train lines: 2510 | val lines: 0
lines starting with val_ inside train.txt: 420
first train line: train_Female_01110|Female|{స ్ వ ర ్ ణ క ా ర ు డ ు}|స్వర్ణ కారుడు
{"Male": 0, "Female": 1}


In [ ]:
import os, shutil
P = "/content/preprocessed_data/Telugu"

lines = [l.rstrip("\n") for l in open(f"{P}/train.txt", encoding="utf8") if l.strip()]
lines += [l.rstrip("\n") for l in open(f"{P}/val.txt", encoding="utf8") if l.strip()]

if not os.path.exists(f"{P}/all.txt"):
    open(f"{P}/all.txt", "w", encoding="utf8").write("\n".join(lines) + "\n")

train = [l for l in lines if l.startswith("train_")]
val   = [l for l in lines if l.startswith("val_")]
assert len(train) + len(val) == len(lines), "some lines start with neither train_ nor val_"

open(f"{P}/train.txt", "w", encoding="utf8").write("\n".join(train) + "\n")
open(f"{P}/val.txt", "w", encoding="utf8").write("\n".join(val) + "\n")
print("train:", len(train), "| val:", len(val))
print("overlap:", {l.split('|')[0] for l in train} & {l.split('|')[0] for l in val})

train: 2090 | val: 420
overlap: set()


## 23. Verify the preprocessed data



In [ ]:
import numpy as np
P = "/content/preprocessed_data/Telugu"
tr = [l.rstrip("\n") for l in open(f"{P}/train.txt", encoding="utf8")]
va = [l.rstrip("\n") for l in open(f"{P}/val.txt", encoding="utf8")]
print(len(tr), len(va), "overlap:", {l.split('|')[0] for l in tr} & {l.split('|')[0] for l in va})
def hours(lines):
    return sum(np.load(f"{P}/mel/{l.split('|')[1]}-mel-{l.split('|')[0]}.npy", mmap_mode="r").shape[0] for l in lines) * 256 / 22050 / 3600
print("train h:", round(hours(tr), 2), "| val h:", round(hours(va), 2))
print(open(f"{P}/stats.json").read()); print(open(f"{P}/speakers.json").read())
print("first training line:", tr[0])

2090 420 overlap: set()
train h: 3.44 | val h: 0.69
{"pitch": [-4.346359024267834, 11.274387005228208, 197.87840232537087, 45.52739458947584], "energy": [-1.2175460025806941, 5.7428349656579645, 43.23497909502608, 35.50993474035954]}
{"Male": 0, "Female": 1}
first training line: train_Female_01110|Female|{స ్ వ ర ్ ణ క ా ర ు డ ు}|స్వర్ణ కారుడు


In [ ]:
!rm -f /content/drive/MyDrive/fs2/preprocessed_Telugu.zip
!cd /content && zip -rq /content/drive/MyDrive/fs2/preprocessed_Telugu.zip preprocessed_data/Telugu -x "preprocessed_data/Telugu/TextGrid/*"
!ls -lh /content/drive/MyDrive/fs2/preprocessed_Telugu.zip

-rw------- 1 root root 350M Nov  7 01:22 /content/drive/MyDrive/fs2/preprocessed_Telugu.zip


## 24. Save the preprocessed data to Drive

Zip the preprocessed features (without TextGrids) so they can be reused in a new session.

In [ ]:
!cd /content && zip -rq /content/drive/MyDrive/fs2/preprocessed_Telugu.zip preprocessed_data/Telugu -x "preprocessed_data/Telugu/TextGrid/*"
!ls -lh /content/drive/MyDrive/fs2/preprocessed_Telugu.zip

## 25. Training setup and code fixes


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

%cd /content
!rm -rf project
!git clone https://github.com/Narmadas-31/fastspeech2-english-telugu.git project

# data and vocoder
!unzip -q -o /content/drive/MyDrive/fs2/preprocessed_Telugu.zip -d /content/
!mkdir -p /tmp/voc && unzip -q -o /content/drive/MyDrive/fs2/generator_universal.pth.tar.zip -d /tmp/voc
!mkdir -p /content/project/Telugu/hifigan
!find /tmp/voc -name generator_universal.pth.tar -exec cp {} /content/project/Telugu/hifigan/ \;

%cd /content/project/Telugu
!pip install -q inflect g2p_en unidecode pyworld tgt librosa scipy matplotlib tensorboard tqdm pyyaml

# config + symbols (from section 20)
!python /content/drive/MyDrive/fs2/patch_telugu_config.py
!python /content/drive/MyDrive/fs2/patch_telugu_symbols.py
!test -f apply_patches.py && python apply_patches.py || echo "no apply_patches.py in Telugu folder - skipped"

# --- code fixes (until you push the fixed files to GitHub) ---
import re
s = open("dataset.py", encoding="utf-8").read()
for k in ["pitch", "energy", "duration"]:
    s = s.replace(f"        {k} = np.load({k}_path).astype(np.float32)\n", "", 1)
s = s.replace("from utils.utils import to_device", "from utils.tools import to_device")
s = re.sub(
    r'preprocess_config = yaml\.load\(\s*open\("\./config/LJSpeech/preprocess\.yaml", "r"\), Loader=yaml\.FullLoader\s*\)\s*'
    r'train_config = yaml\.load\(\s*open\("\./config/LJSpeech/train\.yaml", "r"\), Loader=yaml\.FullLoader\s*\)',
    'cfg = yaml.load(open("config.yaml", "r"), Loader=yaml.FullLoader)\n'
    '    preprocess_config = cfg["preprocessing"]\n    train_config = cfg["train"]', s)
open("dataset.py", "w", encoding="utf-8").write(s)

m = open("utils/model.py", encoding="utf-8").read()
m = m.replace("torch.load(ckpt_path)", "torch.load(ckpt_path, map_location=device, weights_only=False)")
m = m.replace('torch.load("hifigan/generator_LJSpeech.pth.tar")', 'torch.load("hifigan/generator_LJSpeech.pth.tar", map_location=device, weights_only=False)')
m = m.replace('torch.load("hifigan/generator_universal.pth.tar")', 'torch.load("hifigan/generator_universal.pth.tar", map_location=device, weights_only=False)')
m = m.replace("model.requires_grad_ = False", "model.requires_grad_(False)")
open("utils/model.py", "w", encoding="utf-8").write(m)

t = open("utils/tools.py", encoding="utf-8").read()
t = t.replace("torch.from_numpy(energies).to(device)", "torch.from_numpy(energies).float().to(device)")
open("utils/tools.py", "w", encoding="utf-8").write(t)

# test config (separate output folder, nothing on Drive is touched)
import yaml, torch
print("GPU:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
cfg = yaml.safe_load(open("config.yaml"))
out = "/content/test_output_te"
cfg["train"]["path"] = {"ckpt_path": f"{out}/ckpt", "log_path": f"{out}/log", "result_path": f"{out}/result"}
cfg["train"]["step"].update(total_step=200, log_step=10, synth_step=100, val_step=100, save_step=100)
yaml.safe_dump(cfg, open("config_test.yaml", "w"), sort_keys=False, allow_unicode=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content
Cloning into 'project'...
remote: Enumerating objects: 111, done.
remote: Counting objects: 100% (111/111), done.
remote: Compressing objects: 100% (80/80), done.
remote: Total 111 (delta 31), reused 105 (delta 27), pack-reused 0 (from 0)
Receiving objects: 100% (111/111), 1.40 MiB | 12.02 MiB/s, done.
Resolving deltas: 100% (31/31), done.
/content/project/Telugu
Telugu config.yaml was empty -> using the English config as the base
dataset: Telugu
path:
  corpus_path: /content/raw_data/Telugu
  lexicon_path: /content/drive/MyDrive/fs2/backup/telugu_dict.txt
  raw_path: /content/raw_data/Telugu
  preprocessed_path: /content/preprocessed_data/Telugu
preprocessing:
  val_size: 0
  text:
    text_cleaners:
    - basic_cleaners
    language: te
  audio:
    sampling_rate: 22050
    max_wav_value: 32768.0
  stft:
    filter_length: 1024
    hop_length: 256


## 26. Smoke test: short training run

Run `dataset.py` and train for 200 steps to confirm that the whole pipeline works before full training. **If this fails, stop and send me the red error text.**

In [ ]:
!mkdir -p /content/drive/MyDrive/fs2/logs
!python dataset.py
!python train.py config_test.yaml 2>&1 | tee /content/drive/MyDrive/fs2/logs/telugu_smoke_test.log

# save the test checkpoints to Drive automatically
!mkdir -p /content/drive/MyDrive/fs2/output/Telugu/ckpt
!cp -n /content/test_output*/ckpt/*.pth.tar /content/drive/MyDrive/fs2/output/Telugu/ckpt/
!ls -lh /content/drive/MyDrive/fs2/output/Telugu/ckpt


Training set with size 2090 is composed of 130 batches.
Validation set with size 420 is composed of 27 batches.
I0000 00:00:1794014900.577676   41094 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1794014902.787698   41094 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
/usr/local/lib/python3.13/dist-packages/torch/nn/utils/weight_norm.py:145: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)
Prepare training ...
Number of FastSpeech2 Parameters: 35432001
Removing weight norm...
Training:   4%|▍         | 9/200 [00:06<01:41,  1.87it/s]
                                                          
Training:   8%|▊         |

In [ ]:
import re
p = "/content/project/Telugu/dataset.py"
s = open(p, encoding="utf-8").read()

for k in ["pitch", "energy", "duration"]:
    if f"{k} = np.load(" in s:
        print(k, "already present")
        continue
    line = f"        {k} = np.load({k}_path)" + (".astype(np.float32)" if k != "duration" else "") + "\n"
    pat = re.compile(rf"(        {k}_path = os\.path\.join\(.*?\n        \)\n)", re.S)
    s, n = pat.subn(lambda m: m.group(1) + line, s, count=1)
    print(k, "restored" if n else "COULD NOT RESTORE")

open(p, "w", encoding="utf-8").write(s)

pitch restored
energy restored
duration already present


7921

In [ ]:
!grep -n "np.load" /content/project/Telugu/dataset.py

44:        mel = np.load(mel_path).astype(np.float32)
51:        pitch = np.load(pitch_path).astype(np.float32)
58:        energy = np.load(energy_path).astype(np.float32)
65:        duration = np.load(duration_path)


## 27. Copy test checkpoints to Drive

Save the 100- and 200-step checkpoints to Drive so full training can resume from them.

In [ ]:
import os
import shutil

source = "/content/test_output_te/ckpt"
destination = "/content/drive/MyDrive/fs2/output/Telugu/ckpt"

os.makedirs(destination, exist_ok=True)

for filename in ["100.pth.tar", "200.pth.tar"]:
    src = os.path.join(source, filename)
    dst = os.path.join(destination, filename)

    if os.path.exists(src) and not os.path.exists(dst):
        shutil.copy2(src, dst)
        print("Copied:", filename)
    elif os.path.exists(dst):
        print("Already exists:", filename)
    else:
        print("Missing:", filename)

Already exists: 100.pth.tar
Already exists: 200.pth.tar


## 28. Start full training

Train the model with `config.yaml` (10000 steps in total), resuming from step 200.

In [ ]:
import os, subprocess
B = "/content/drive/MyDrive/fs2"
P = "/content/preprocessed_data/Telugu"
Z = f"{B}/preprocessed_Telugu.zip"

print("Drive mounted:", os.path.ismount("/content/drive"))
print("project folder:", os.path.isdir("/content/project/Telugu"))
print("train.txt exists:", os.path.exists(f"{P}/train.txt"))
zip_mb = round(os.path.getsize(Z) / 1e6) if os.path.exists(Z) else 0
print("zip on Drive (MB):", zip_mb)

if os.path.exists(f"{P}/train.txt"):
    print("-> Data is already here. Start training.")
elif os.path.ismount("/content/drive") and zip_mb > 100:
    subprocess.run(f"unzip -q -o {Z} -d /content/", shell=True)
    print("-> Restored from the zip. train.txt exists now:", os.path.exists(f"{P}/train.txt"))
else:
    print("-> No usable data. You must run the rebuild cell first (the long one starting with 'import os, re, glob, json, shutil, yaml').")

Drive mounted: True
project folder: True
train.txt exists: False
zip on Drive (MB): 0
-> No usable data. You must run the rebuild cell first (the long one starting with 'import os, re, glob, json, shutil, yaml').


In [ ]:
%cd /content/project/Telugu

!mkdir -p /content/drive/MyDrive/fs2/logs
!python train.py config.yaml --restore_step 200 2>&1 | tee -a /content/drive/MyDrive/fs2/logs/telugu_train.log

/content/project/Telugu
I0000 00:00:1794022132.973545   24642 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1794022133.114614   24642 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1794022137.560746   24642 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
Prepare training ...
Traceback (most recent call last):
  File "/content/project/Telugu/train.py", line 179, in <module>
    main(args, configs)
    ~~~~^^^^^^^^^^^^^^^
  File "/content/project/Telugu/train.py", line 27, in main
    dataset = Dataset(
        "train.txt", preprocess_config, train_config, sort=True, drop_last=True
    )
  File "/content/project/Telugu/dataset.py", line 21, in __init__
    self.basename, self.speak

In [ ]:
import os, shutil
from google.colab import drive

if not os.path.ismount("/content/drive"):
    shutil.rmtree("/content/drive", ignore_errors=True)  # removes only a leftover local folder, never your real Drive
    drive.mount("/content/drive")

print("Drive mounted:", os.path.ismount("/content/drive"))
B = "/content/drive/MyDrive/fs2"
for p in ["backup/raw_Telugu.zip", "preprocessed_data/Telugu/TextGrid/Female",
          "preprocessed_data/Telugu/TextGrid/Male", "generator_universal.pth.tar.zip",
          "backup/telugu_dict.txt"]:
    print("FOUND  " if os.path.exists(f"{B}/{p}") else "MISSING", p)

Mounted at /content/drive
Drive mounted: True
FOUND   backup/raw_Telugu.zip
FOUND   preprocessed_data/Telugu/TextGrid/Female
FOUND   preprocessed_data/Telugu/TextGrid/Male
FOUND   generator_universal.pth.tar.zip
FOUND   backup/telugu_dict.txt


In [ ]:
import os
B = "/content/drive/MyDrive/fs2"
assert os.path.ismount("/content/drive"), "Drive is not mounted"
for p in ["backup/raw_Telugu.zip", "preprocessed_data/Telugu/TextGrid/Female",
          "preprocessed_data/Telugu/TextGrid/Male", "generator_universal.pth.tar.zip",
          "backup/telugu_dict.txt"]:
    assert os.path.exists(f"{B}/{p}"), f"MISSING on Drive: {p} - stop and tell me"

%cd /content
!rm -rf project
!git clone https://github.com/Narmadas-31/fastspeech2-english-telugu.git project
!pip install -q librosa soundfile pyyaml tgt pyworld g2p_en inflect unidecode scikit-learn tensorboard matplotlib pandas scipy
!mkdir -p /tmp/voc && unzip -q -o {B}/generator_universal.pth.tar.zip -d /tmp/voc
!mkdir -p /content/project/Telugu/hifigan
!find /tmp/voc -name generator_universal.pth.tar -exec cp {} /content/project/Telugu/hifigan/ \;
!mkdir -p /content/raw_data /content/preprocessed_data/Telugu
!unzip -q -o {B}/backup/raw_Telugu.zip -d /content/
!cp -r {B}/preprocessed_data/Telugu/TextGrid /content/preprocessed_data/Telugu/
print("wav files:", sum(f.endswith(".wav") for _, _, fs in os.walk("/content/raw_data/Telugu") for f in fs))
print("TextGrids:", sum(f.endswith(".TextGrid") for _, _, fs in os.walk("/content/preprocessed_data/Telugu/TextGrid") for f in fs))

/content
Cloning into 'project'...
remote: Enumerating objects: 111, done.
remote: Counting objects: 100% (111/111), done.
remote: Compressing objects: 100% (80/80), done.
remote: Total 111 (delta 31), reused 105 (delta 27), pack-reused 0 (from 0)
Receiving objects: 100% (111/111), 1.40 MiB | 6.47 MiB/s, done.
Resolving deltas: 100% (31/31), done.
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
wav files: 2533
TextGrids: 2510


In [ ]:
import os, torch
B = "/content/drive/MyDrive/fs2"
T = "/content/project/Telugu"
P = "/content/preprocessed_data/Telugu"
assert os.path.ismount("/content/drive"), "Drive is not mounted"
assert os.path.isdir(f"{P}/TextGrid"), "Data is missing - run the first cell (clone + restore data) again"

# ---- 1) restore the wiped file from the repo, then fix it safely ----
!cd /content/project && git checkout -- Telugu/preprocessor/preprocessor.py
pf = f"{T}/preprocessor/preprocessor.py"
src = open(pf, encoding="utf8").read()
assert "class Preprocessor" in src, "preprocessor.py is still empty or wrong - tell me"
open(pf, "w", encoding="utf8").write(src.replace("tier._objects", "tier.intervals"))
print("preprocessor.py restored:", len(src), "characters")

# ---- 2) preprocess ----
%cd {T}
!python preprocess.py

# ---- 3) split train / val by file name ----
lines = [l.rstrip("\n") for l in open(f"{P}/train.txt", encoding="utf8") if l.strip()]
lines += [l.rstrip("\n") for l in open(f"{P}/val.txt", encoding="utf8") if l.strip()]
train = [l for l in lines if l.startswith("train_")]
val = [l for l in lines if l.startswith("val_")]
assert len(train) + len(val) == len(lines)
open(f"{P}/train.txt", "w", encoding="utf8").write("\n".join(train) + "\n")
open(f"{P}/val.txt", "w", encoding="utf8").write("\n".join(val) + "\n")
print("train:", len(train), "| val:", len(val), "| overlap:", {l.split('|')[0] for l in train} & {l.split('|')[0] for l in val})

# ---- 4) zip to Drive ----
assert os.path.ismount("/content/drive")
!rm -f {B}/preprocessed_Telugu.zip
!cd /content && zip -rq {B}/preprocessed_Telugu.zip preprocessed_data/Telugu -x "preprocessed_data/Telugu/TextGrid/*"
!ls -lh {B}/preprocessed_Telugu.zip
print("GPU:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

preprocessor.py restored: 10912 characters
/content/project/Telugu
/usr/local/lib/python3.13/dist-packages/pyworld/__init__.py:13: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
Processing Data ...
  0% 0/2 [00:00<?, ?it/s]
Traceback (most recent call last):
  File "/content/project/Telugu/preprocess.py", line 5, in <module>
    Preprocessor(cfg['preprocessing']).build_from_path()
    ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^
  File "/content/project/Telugu/preprocessor/preprocessor.py", line 78, in build_from_path
    ret = self.process_utterance(speaker, basename)
  File "/content/project/Telugu/preprocessor/preprocessor.py", line 194, in process_utterance
    mel_spectrogram, energy = Audio.tools.get_mel_from_wav(wav, self.STFT)
                             

FileNotFoundError: [Errno 2] No such file or directory: '/content/preprocessed_data/Telugu/train.txt'

## 29. Resume training after a disconnect



In [ ]:
RESTORE_STEP = 6000
%cd /content/project/Telugu
!mkdir -p /content/drive/MyDrive/fs2/logs
!python train.py config.yaml --restore_step {RESTORE_STEP} 2>&1 | tee -a /content/drive/MyDrive/fs2/logs/telugu_train.log

/content/project/Telugu
I0000 00:00:1794040944.339577   16608 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1794040946.816921   16608 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1794040953.949748   16608 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
Prepare training ...
Traceback (most recent call last):
  File "/content/project/Telugu/train.py", line 179, in <module>
    main(args, configs)
    ~~~~^^^^^^^^^^^^^^^
  File "/content/project/Telugu/train.py", line 27, in main
    dataset = Dataset(
        "train.txt", preprocess_config, train_config, sort=True, drop_last=True
    )
  File "/content/project/Telugu/dataset.py", line 21, in __init__
    self.basename, self.speak

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

%cd /content
!rm -rf project
!git clone https://github.com/Narmadas-31/fastspeech2-english-telugu.git project

# data and vocoder
!unzip -q -o /content/drive/MyDrive/fs2/preprocessed_Telugu.zip -d /content/
!mkdir -p /tmp/voc && unzip -q -o /content/drive/MyDrive/fs2/generator_universal.pth.tar.zip -d /tmp/voc
!mkdir -p /content/project/Telugu/hifigan
!find /tmp/voc -name generator_universal.pth.tar -exec cp {} /content/project/Telugu/hifigan/ \;

%cd /content/project/Telugu
!pip install -q inflect g2p_en unidecode pyworld tgt librosa scipy matplotlib tensorboard tqdm pyyaml

# config + symbols (from section 20)
!python /content/drive/MyDrive/fs2/patch_telugu_config.py
!python /content/drive/MyDrive/fs2/patch_telugu_symbols.py
!test -f apply_patches.py && python apply_patches.py || echo "no apply_patches.py in Telugu folder - skipped"

# --- code fixes (until you push the fixed files to GitHub) ---
import re
s = open("dataset.py", encoding="utf-8").read()
for k in ["pitch", "energy", "duration"]:
    s = s.replace(f"        {k} = np.load({k}_path).astype(np.float32)\n", "", 1)
s = s.replace("from utils.utils import to_device", "from utils.tools import to_device")
s = re.sub(
    r'preprocess_config = yaml\.load\(\s*open\("\./config/LJSpeech/preprocess\.yaml", "r"\), Loader=yaml\.FullLoader\s*\)\s*'
    r'train_config = yaml\.load\(\s*open\("\./config/LJSpeech/train\.yaml", "r"\), Loader=yaml\.FullLoader\s*\)',
    'cfg = yaml.load(open("config.yaml", "r"), Loader=yaml.FullLoader)\n'
    '    preprocess_config = cfg["preprocessing"]\n    train_config = cfg["train"]', s)
open("dataset.py", "w", encoding="utf-8").write(s)

m = open("utils/model.py", encoding="utf-8").read()
m = m.replace("torch.load(ckpt_path)", "torch.load(ckpt_path, map_location=device, weights_only=False)")
m = m.replace('torch.load("hifigan/generator_LJSpeech.pth.tar")', 'torch.load("hifigan/generator_LJSpeech.pth.tar", map_location=device, weights_only=False)')
m = m.replace('torch.load("hifigan/generator_universal.pth.tar")', 'torch.load("hifigan/generator_universal.pth.tar", map_location=device, weights_only=False)')
m = m.replace("model.requires_grad_ = False", "model.requires_grad_(False)")
open("utils/model.py", "w", encoding="utf-8").write(m)

t = open("utils/tools.py", encoding="utf-8").read()
t = t.replace("torch.from_numpy(energies).to(device)", "torch.from_numpy(energies).float().to(device)")
open("utils/tools.py", "w", encoding="utf-8").write(t)

# test config (separate output folder, nothing on Drive is touched)
import yaml, torch
print("GPU:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
cfg = yaml.safe_load(open("config.yaml"))
out = "/content/test_output_te"
cfg["train"]["path"] = {"ckpt_path": f"{out}/ckpt", "log_path": f"{out}/log", "result_path": f"{out}/result"}
cfg["train"]["step"].update(total_step=200, log_step=10, synth_step=100, val_step=100, save_step=100)
yaml.safe_dump(cfg, open("config_test.yaml", "w"), sort_keys=False, allow_unicode=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content
Cloning into 'project'...
remote: Enumerating objects: 111, done.
remote: Counting objects: 100% (111/111), done.
remote: Compressing objects: 100% (80/80), done.
remote: Total 111 (delta 31), reused 105 (delta 27), pack-reused 0 (from 0)
Receiving objects: 100% (111/111), 1.40 MiB | 8.41 MiB/s, done.
Resolving deltas: 100% (31/31), done.
unzip:  cannot find or open /content/drive/MyDrive/fs2/preprocessed_Telugu.zip, /content/drive/MyDrive/fs2/preprocessed_Telugu.zip.zip or /content/drive/MyDrive/fs2/preprocessed_Telugu.zip.ZIP.
/content/project/Telugu
Telugu config.yaml was empty -> using the English config as the base
config patched: language = te | cleaners = ['basic_cleaners']
63 Telugu symbols
no apply_patches.py in Telugu folder - skipped
GPU: False 


## 30. Plot training and validation losses

Generate and display the loss curves from the training logs.

In [ ]:
%cd /content/project/Telugu
!python plot_losses.py config.yaml ../Results/Telugu
!mkdir -p /content/drive/MyDrive/fs2/output/Telugu/loss_plots
!cp ../Results/Telugu/*.png /content/drive/MyDrive/fs2/output/Telugu/loss_plots/
from IPython.display import Image, display
for f in ["training_loss", "validation_loss", "train_vs_val_total"]:
    display(Image(f"../Results/Telugu/{f}.png"))

## 31. Inference with different checkpoints

Synthesise the five Telugu test sentences (T1, T2a, T2b, T3, T4) with checkpoints at 2000, 6000 and 10000 steps.
Set `SPEAKER_ID` to one of the ids printed from `speakers.json` in section 23.

In [ ]:
import os
os.environ["SPEAKER_ID"] = "0"

In [ ]:
%%bash
cd /content/project/Telugu
for s in 2000 6000 10000; do
  python inference.py config.yaml --restore_step $s --speaker_id $SPEAKER_ID --out /content/cmp_te/ck$s
done
ls /content/cmp_te/ck10000/test_generated_audios
ls /content/cmp_te/ck10000/variance_adaptor_plots/*
mkdir -p /content/drive/MyDrive/fs2/output/Telugu
rm -rf /content/drive/MyDrive/fs2/output/Telugu/cmp
cp -r /content/cmp_te /content/drive/MyDrive/fs2/output/Telugu/cmp
echo "saved to Drive: fs2/output/Telugu/cmp"


## 32. Listen to the generated audio

Play the audio from each checkpoint for all five test sentences.

In [ ]:
from IPython.display import Audio, display
for s in [2000, 6000, 10000]:
    for sid in ["T1", "T2a", "T2b", "T3", "T4"]:
        print(f"checkpoint {s} - {sid}")
        display(Audio(f"/content/cmp_te/ck{s}/test_generated_audios/{sid}.wav"))

## 33. Variance adaptor plots

Display the predicted duration, pitch and energy plots for the first test sentence.

In [ ]:
from IPython.display import Image, display
for k in ["duration", "pitch", "energy"]:
    print(k)
    display(Image(f"/content/cmp_te/ck10000/variance_adaptor_plots/{k}/T1.png"))

## 34. Collect final results



In [ ]:
BEST = 10000
import os, shutil, torch
%cd /content/project/Telugu
os.makedirs("/content/project/Results", exist_ok=True)
shutil.rmtree("/content/project/Results/Telugu", ignore_errors=True)
shutil.copytree(f"/content/cmp_te/ck{BEST}", "/content/project/Results/Telugu")
!python plot_losses.py config.yaml ../Results/Telugu
!mkdir -p /content/drive/MyDrive/fs2/output/Telugu/loss_plots
!cp ../Results/Telugu/*.png /content/drive/MyDrive/fs2/output/Telugu/loss_plots/
!ls /content/project/Results/Telugu

# weights-only checkpoint for GitHub (the full one stays on Drive)
full = f"/content/drive/MyDrive/fs2/output/Telugu/ckpt/{BEST}.pth.tar"
os.makedirs("/content/project/Telugu/checkpoints", exist_ok=True)
slim = f"/content/project/Telugu/checkpoints/{BEST}.pth.tar"
ck = torch.load(full, map_location="cpu", weights_only=False)
torch.save({"model": ck["model"]}, slim)
print("full MB:", round(os.path.getsize(full)/1e6), "| slim MB:", round(os.path.getsize(slim)/1e6))

## 35. Save everything to Google Drive


What is where on Drive after this cell:
- `fs2/output/Telugu/ckpt/` : all full checkpoints (saved automatically during training)
- `fs2/output/Telugu/log/` : training logs (used for the loss plots)
- `fs2/output/Telugu/cmp/` : audio and plots from every checkpoint
- `fs2/Submission/Results/Telugu/` : final results (loss plots, 5 test audios, duration/pitch/energy plots)
- `fs2/Submission/Telugu/checkpoints/` : the small checkpoint for GitHub

In [ ]:
import os, shutil
SUB = "/content/drive/MyDrive/fs2/Submission"
shutil.rmtree(f"{SUB}/Results/Telugu", ignore_errors=True)
os.makedirs(f"{SUB}/Results", exist_ok=True)
shutil.copytree("/content/project/Results/Telugu", f"{SUB}/Results/Telugu")
os.makedirs(f"{SUB}/Telugu/checkpoints", exist_ok=True)
shutil.copy2(f"/content/project/Telugu/checkpoints/{BEST}.pth.tar", f"{SUB}/Telugu/checkpoints/{BEST}.pth.tar")
!find /content/drive/MyDrive/fs2/Submission -type f | sort
!ls -lh /content/drive/MyDrive/fs2/output/Telugu/ckpt